In [ ]:
import sys
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, LeaveOneOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.svm import SVC
from code.Data_Preprocessing import DataPreprocessing_melt as mf
from code.Data_Processing import TargetVariables as tv
from code.Data_Processing import DataProcessing_run as rf
from joblib import Parallel, delayed

In [ ]:
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  
lower_pct = 40
upper_pct = 60
# Load MFCC.pkl file 
df = pd.read_pickle("/planilhas/MFCC35.pkl")
print(f"Patients: {df['Patient_ID'].nunique()}")

____

- Note: For calculating the standardized Y values using HDRS/CDI raw score data, refer to the worksheet in the data folder of this project's main DOI. [Use "Patient_ID" as a cross-reference.]

B1 - YA + A (Y = Delta_Y : Worse and Better) MFCCs All sessions

In [ ]:
df_model_YAA_S3_B = df.copy()
df_model_YAA_S3_B = tv.standardized_binary_evolution(df_model_YAA_S3_B, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'] = df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'].map({'Worse': 0, 'Better': 1})
df_model_YAA_S3_B = df_model_YAA_S3_B.dropna(subset=['Y_Binary_Classe_Delta_Y'])
df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'] = df_model_YAA_S3_B['Y_Binary_Classe_Delta_Y'].astype(int)

In [ ]:
meta_cols_YAA = ['Patient_ID', 'Y_Binary_Classe_Delta_Y']
df_model_YAA_S3_B = mf.get_mfccs_per_segment_mean_std(df_model_YAA_S3_B, meta_cols_YAA)
df_model_YAA_S3_B = df_model_YAA_S3_B.dropna()
print(f"Patients: {df_model_YAA_S3_B['Patient_ID'].nunique()}")

In [ ]:
def global_patient_speech_prop(df_melt, segment_len_s=5.0, total_audio_all_s=996_000,
                               patient_col='Patient_ID', session_col='Session', segment_col='Segment'):
    seg_keys = [patient_col, session_col, segment_col]
    uniq_segs = df_melt[seg_keys].drop_duplicates()
    patient_total_s = float(len(uniq_segs) * segment_len_s)
    prop = patient_total_s / float(total_audio_all_s)
    return {
        "n_unique_segments": len(uniq_segs),
        "patient_total_seconds": patient_total_s,
        "patient_total_hours": patient_total_s / 3600.0,
        "prop_patient_overall": prop,           # 0..1
        "prop_patient_overall_pct": prop * 100  # %
    }
def per_patient_counts(df_melt, segment_len_s=5.0,
                       patient_col='Patient_ID', session_col='Session', segment_col='Segment'):
    seg_keys = [patient_col, session_col, segment_col]
    uniq_segs = df_melt[seg_keys].drop_duplicates()
    per_pat = (uniq_segs
               .groupby(patient_col)
               .size()
               .rename('n_segments')
               .to_frame())
    per_pat['speech_seconds'] = per_pat['n_segments'] * segment_len_s
    per_pat['speech_hours'] = per_pat['speech_seconds'] / 3600.0
    return per_pat.reset_index()
tot = global_patient_speech_prop(df_model_YAA_S3_B, segment_len_s=5.0, total_audio_all_s=996_000)
print(tot)
per_pat = per_patient_counts(df_model_YAA_S3_B, segment_len_s=5.0)
print(per_pat.head())

B3 - YA (Y = Delta_HDRS : Worse and Better) MFCCs All sessions

In [ ]:
df_model_YA_S3_B = df.copy()
df_model_YA_S3_B = tv.standardized_binary_evolution_HDRS(df_model_YA_S3_B, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_YA_S3_B['Y_Binary_Classe_HDRS'] = df_model_YA_S3_B['Y_Binary_Classe_HDRS'].map({'Worse': 0, 'Better': 1})
df_model_YA_S3_B = df_model_YA_S3_B.dropna(subset=['Y_Binary_Classe_HDRS'])
df_model_YA_S3_B['Y_Binary_Classe_HDRS'] = df_model_YA_S3_B['Y_Binary_Classe_HDRS'].astype(int)

In [ ]:
meta_cols_YA = ['Patient_ID', 'Y_Binary_Classe_HDRS']
df_model_YA_S3_B = mf.get_mfccs_per_segment_mean_std(df_model_YA_S3_B, meta_cols_YA)
df_model_YA_S3_B = df_model_YA_S3_B.dropna()
print(f"Patients: {df_model_YA_S3_B['Patient_ID'].nunique()}")

B5 - A (Y = Delta_CDI : Worse and Better) MFCCs All sessions

In [ ]:
df_model_A_S3_B = df.copy()
df_model_A_S3_B = tv.standardized_binary_evolution_CDI(df_model_A_S3_B, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
df_model_A_S3_B['Y_Binary_Classe_CDI'] = df_model_A_S3_B['Y_Binary_Classe_CDI'].map({'Worse': 0, 'Better': 1})
df_model_A_S3_B = df_model_A_S3_B.dropna(subset=['Y_Binary_Classe_CDI'])
df_model_A_S3_B['Y_Binary_Classe_CDI'] = df_model_A_S3_B['Y_Binary_Classe_CDI'].astype(int)

In [ ]:
meta_cols_A = ['Patient_ID', 'Y_Binary_Classe_CDI']
df_model_A_S3_B = mf.get_mfccs_per_segment_mean_std(df_model_A_S3_B, meta_cols_A)
df_model_A_S3_B = df_model_A_S3_B.dropna()
print(f"Patients: {df_model_A_S3_B['Patient_ID'].nunique()}")

In [ ]:
datasets = [
    (df_model_YAA_S3_B, 'Y_Binary_Classe_Delta_Y'),
    (df_model_YA_S3_B, 'Y_Binary_Classe_HDRS'),
    (df_model_A_S3_B, 'Y_Binary_Classe_CDI')]

______

Running the experiments (LOO-CV patient-independent)

In [ ]:
summary_results = []
all_results = []
for idx, (df, target_column) in enumerate(datasets):
    print(f"\nProcessing Dataset {idx+1} - Target: {target_column}")
    unique_patients = df['Patient_ID'].unique()    
    results_rf = Parallel(n_jobs=64, backend='loky')(
        delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, RandomForestClassifier(class_weight='balanced', random_state=42), rf.class_param_grid_rf)
        for patient in unique_patients)
    print("RF Done")
    result_lr = Parallel(n_jobs=32)(
        delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, LogisticRegression(class_weight='balanced', random_state=42), rf.class_param_grid_logreg)
        for patient in unique_patients)
    print("LogR Done")
    results_xgb = Parallel(n_jobs=4)(
        delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, xgb.XGBClassifier(n_jobs=1, random_state=42, use_label_encoder=False, eval_metric='mlogloss'), rf.class_param_grid_xgb)
        for patient in unique_patients)
    print("XGB Done")
    results_mlp = Parallel(n_jobs=64, backend='loky')(
        delayed(rf.class_process_leave_one_out)(patient, df, 'Patient_ID', target_column, MLPClassifier(random_state=42), rf.class_param_grid_mlp)
        for patient in unique_patients)
    print("MLP Done")
    current_results = results_rf + result_lr + results_xgb + results_mlp
    all_results.extend(current_results)   
    df_current = pd.DataFrame(current_results)
    for model_name in df_current["Model"].unique():
        df_model = df_current[df_current["Model"] == model_name]
        summary_results.append({
            "Dataset_Index": idx + 1,
            "Target_Column": target_column,
            "Model": model_name,
            "Recall_Mean":   df_model["Recall"].mean(),
            "F1_Score_Mean": df_model["F1_Score"].mean(),
            "Precision_Mean": df_model["Precision"].mean(),
            "ROC_AUC_Mean": df_model["ROC_AUC"].dropna().mean() if "ROC_AUC" in df_model else None,
            "PR_AUC_Mean":  df_model["PR_AUC"].dropna().mean() if "PR_AUC" in df_model else None})
df_results = pd.DataFrame(all_results)
df_summary = pd.DataFrame(summary_results)